In [1]:
from ucimlrepo import fetch_ucirepo
import pandas as pd 

In [2]:
# fetch dataset 
car_evaluation = fetch_ucirepo(id=19) 
  
# data (as pandas dataframes) 
x = car_evaluation.data.features 
y = car_evaluation.data.targets 
car_evaluation.variables 

,name,role,type,demographic,description,units,missing_values
0,buying,Feature,Categorical,None,buying price,None,no
1,maint,Feature,Categorical,None,price of the maintenance,None,no
2,doors,Feature,Categorical,None,number of doors,None,no
3,persons,Feature,Categorical,None,capacity in terms of persons to carry,None,no
4,lug_boot,Feature,Categorical,None,the size of luggage boot,None,no
5,safety,Feature,Categorical,None,estimated safety of the car,None,no
6,class,Target,Categorical,None,"evaulation level (unacceptable, acceptable, go...",None,no


In [3]:
x.head()

,buying,maint,doors,persons,lug_boot,safety
0,vhigh,vhigh,2,2,small,low
1,vhigh,vhigh,2,2,small,med
2,vhigh,vhigh,2,2,small,high
3,vhigh,vhigh,2,2,med,low
4,vhigh,vhigh,2,2,med,med


In [4]:
y.head()

,class
0,unacc
1,unacc
2,unacc
3,unacc
4,unacc


In [5]:
for i in x:
    print(i,":=",list(x[i].unique()))

print("class :=",list(y['class'].unique()))

buying := ['vhigh', 'high', 'med', 'low']
maint := ['vhigh', 'high', 'med', 'low']
doors := ['2', '3', '4', '5more']
persons := ['2', '4', 'more']
lug_boot := ['small', 'med', 'big']
safety := ['low', 'med', 'high']
class := ['unacc', 'acc', 'vgood', 'good']


In [6]:
# preprocessing of data
# converting data into transaction format

dataset=x
dataset['class']=y['class']
encoded_dataset = pd.get_dummies(dataset)
encoded_dataset

,buying_high,buying_low,buying_med,buying_vhigh,maint_high,maint_low,maint_med,maint_vhigh,doors_2,doors_3,...,lug_boot_big,lug_boot_med,lug_boot_small,safety_high,safety_low,safety_med,class_acc,class_good,class_unacc,class_vgood
0,False,False,False,True,False,False,False,True,True,False,...,False,False,True,False,True,False,False,False,True,False
1,False,False,False,True,False,False,False,True,True,False,...,False,False,True,False,False,True,False,False,True,False
2,False,False,False,True,False,False,False,True,True,False,...,False,False,True,True,False,False,False,False,True,False
3,False,False,False,True,False,False,False,True,True,False,...,False,True,False,False,True,False,False,False,True,False
4,False,False,False,True,False,False,False,True,True,False,...,False,True,False,False,False,True,False,False,True,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1723,False,True,False,False,False,True,False,False,False,False,...,False,True,False,False,False,True,False,True,False,False
1724,False,True,False,False,False,True,False,False,False,False,...,False,True,False,True,False,False,False,False,False,True
1725,False,True,False,False,False,True,False,False,False,False,...,True,False,False,False,True,False,False,False,True,False
1726,False,True,False,False,False,True,False,False,False,False,...,True,False,False,False,False,True,False,True,False,False


<H1>Frequent ItemSet using Apriori Method</h1>

In [ ]:
def dataframe_to_transactions(df):
    transactions = []
    
    for _, row in df.iterrows():
        transaction = set(df.columns[row == 1])
        transactions.append(transaction)
    
    return transactions

def get_support(transactions, itemset):
    count = 0
    
    for transaction in transactions:
        if itemset.issubset(transaction):
            count += 1
            
    return count / len(transactions)

def generate_C1(transactions):
    items = set()
    
    for transaction in transactions:
        for item in transaction:
            items.add(frozenset([item]))
    
    return items

# Prune by Support → Lk
def filter_candidates(transactions, candidates, min_support):
    frequent = {}
    
    for itemset in candidates:
        support = get_support(transactions, itemset)
        
        if support >= min_support:
            frequent[itemset] = support
            
    return frequent

# Core Apriori logic
def generate_candidates(prev_frequent, k):
    candidates = set()
    prev_items = list(prev_frequent.keys())
    
    for i in range(len(prev_items)):
        for j in range(i + 1, len(prev_items)):
            union = prev_items[i] | prev_items[j]
            
            if len(union) == k:
                candidates.add(union)
    
    return candidates


def apriori(transactions, min_support):
    
    # Step 1: C1
    C1 = generate_C1(transactions)
    # Step 2: L1
    L1 = filter_candidates(transactions, C1, min_support)
    frequent_itemsets = dict(L1)
    
    k = 2
    Lk = L1
    
    while Lk:
        # Step 3: Generate candidates
        Ck = generate_candidates(Lk, k)
        
        # Step 4: Prune
        Lk = filter_candidates(transactions, Ck, min_support)
        
        # Store results
        frequent_itemsets.update(Lk)
        
        k += 1
    
    return frequent_itemsets


transactions = dataframe_to_transactions(encoded_dataset)

frequent_itemsets = apriori(transactions, min_support=0.1)


86

In [22]:
from itertools import combinations

def get_subsets(itemset):
    subsets = []
    for i in range(1, len(itemset)):
        subsets.extend(combinations(itemset, i))
    return list(map(frozenset, subsets))

def generate_rules(frequent_itemsets, min_confidence=0.7):
    
    rules = []
    
    for itemset in frequent_itemsets:
        if len(itemset) < 2:
            continue  # need at least 2 items
        
        subsets = get_subsets(itemset)
        
        for X in subsets:
            Y = itemset - X
            
            if len(Y) == 0:
                continue
            
            support_L = frequent_itemsets[itemset]
            support_X = frequent_itemsets.get(X, 0)
            support_Y = frequent_itemsets.get(Y, 0)
            
            # Avoid division issues
            if support_X == 0:
                continue
            
            confidence = support_L / support_X
            
            if confidence >= min_confidence:
                lift = confidence / support_Y if support_Y > 0 else 0
                
                rules.append({
                    "antecedent": X,
                    "consequent": Y,
                    "support": support_L,
                    "confidence": confidence,
                    "lift": lift
                })
    
    return rules

def filter_class_rules(rules):
    return [
        r for r in rules
        if any('class_' in item for item in r['consequent'])
    ]

rules = generate_rules(frequent_itemsets, min_confidence=0.7)
rules=filter_class_rules(rules)
rules = sorted(
    rules,
    key=lambda x: (x['confidence'], x['support']),
    reverse=True
)
rules


[{'antecedent': frozenset({'persons_2'}),
  'consequent': frozenset({'class_unacc'}),
  'support': 0.3333333333333333,
  'confidence': 1.0,
  'lift': 1.428099173553719},
 {'antecedent': frozenset({'safety_low'}),
  'consequent': frozenset({'class_unacc'}),
  'support': 0.3333333333333333,
  'confidence': 1.0,
  'lift': 1.428099173553719},
 {'antecedent': frozenset({'lug_boot_med', 'safety_low'}),
  'consequent': frozenset({'class_unacc'}),
  'support': 0.1111111111111111,
  'confidence': 1.0,
  'lift': 1.428099173553719},
 {'antecedent': frozenset({'lug_boot_big', 'safety_low'}),
  'consequent': frozenset({'class_unacc'}),
  'support': 0.1111111111111111,
  'confidence': 1.0,
  'lift': 1.428099173553719},
 {'antecedent': frozenset({'lug_boot_small', 'persons_2'}),
  'consequent': frozenset({'class_unacc'}),
  'support': 0.1111111111111111,
  'confidence': 1.0,
  'lift': 1.428099173553719},
 {'antecedent': frozenset({'persons_4', 'safety_low'}),
  'consequent': frozenset({'class_unacc'}